# Chapter 2 — NB 05: reconciliação **VEP/dbNSFP × Biofilter 4**

Comparamos as duas anotações da MESMA tabela união:
- **BF4** (NB 03): DB de variantes conhecidas (gnomAD). `results/03/cmp_union_annotated_bf4.csv`
- **VEP/dbNSFP** (NB 04): anotação por coordenada (todas as variantes). `results/04/cmp_union_annotated_vep.csv`

Perguntas:
1. **Cobertura** — o que o VEP anota que o BF4 perdeu (e o que isso custou).
2. **pLOF** — os dois concordam? quem acha mais?
3. **Consequência** — concordância nas variantes que ambos anotaram.
4. **AlphaMissense** — BF4 (0) vs VEP/dbNSFP.
5. **REVEL** — cross-check dos scores pré-computados (sanidade).

In [1]:
# --- Setup: load both annotated unions, merge on key ---
from pathlib import Path
import pandas as pd, numpy as np

BASE = Path("/project/hall/analysis/hearing-loss-genomics")
R03  = BASE / "analysis/chapter_2/results/03"
R04  = BASE / "analysis/chapter_2/results/04"
R05  = BASE / "analysis/chapter_2/results/05"; R05.mkdir(parents=True, exist_ok=True)

bf4 = pd.read_csv(R03 / "cmp_union_annotated_bf4.csv")
vep = pd.read_csv(R04 / "cmp_union_annotated_vep.csv")

b = bf4[["key","bf4_status","consequence_name","consequence_category","lof_confidence",
         "alphamissense_score","revel_max"]].rename(columns={
         "consequence_name":"bf4_conseq","consequence_category":"bf4_categ",
         "lof_confidence":"bf4_lof","alphamissense_score":"bf4_am","revel_max":"bf4_revel"})
v = vep[["key","CHROM","POS","REF","ALT","group","MAF_v1","MAF_v2",
         "Consequence","IMPACT","is_pLOF_vep","AlphaMissense_score","AlphaMissense_pred"]].rename(columns={
         "Consequence":"vep_conseq","IMPACT":"vep_impact",
         "AlphaMissense_score":"vep_am","AlphaMissense_pred":"vep_am_pred"})
m = v.merge(b, on="key", how="left")
m["maf_max"] = m[["MAF_v1","MAF_v2"]].max(axis=1)
print("union variants:", len(m))
print("bf4_status:", dict(m["bf4_status"].value_counts(dropna=False)))

union variants: 386
bf4_status: {'not_found': np.int64(287), 'found': np.int64(97), 'not_submitted': np.int64(2)}


## 1. Cobertura — o que o VEP anota e o BF4 não

`vep_annot` = VEP deu consequência. `bf4_found` = BF4 achou no DB.

In [2]:
m["vep_annot"] = m["vep_conseq"].notna()
m["bf4_found"] = (m["bf4_status"] == "found")
print(pd.crosstab(m["bf4_found"], m["vep_annot"],
                  rownames=["BF4 found"], colnames=["VEP annotated"]))

# o conjunto que SÓ o VEP cobriu (BF4 perdeu)
only_vep = m[(~m["bf4_found"]) & (m["vep_annot"])]
print(f"\nVEP-only (BF4 não achou): {len(only_vep)} variantes")
print("  consequência (VEP) desse grupo:")
print("   " + only_vep["vep_conseq"].value_counts().head(10).to_string().replace("\n","\n   "))
print(f"  pLOF nesse grupo: {int(only_vep['is_pLOF_vep'].sum())} | com AlphaMissense: {int(only_vep['vep_am'].notna().sum())}")

VEP annotated  False  True 
BF4 found                  
False              2    287
True               0     97

VEP-only (BF4 não achou): 287 variantes
  consequência (VEP) desse grupo:
   vep_conseq
   missense_variant                                                            124
   intron_variant                                                               68
   synonymous_variant                                                           40
   5_prime_UTR_variant                                                          16
   3_prime_UTR_variant                                                          14
   frameshift_variant                                                           11
   stop_gained                                                                   7
   splice_polypyrimidine_tract_variant,intron_variant                            2
   splice_donor_region_variant,intron_variant                                    2
   splice_region_variant,splice_polypyrimidine_tract

## 2. pLOF — BF4 (LOFTEE) vs VEP (IMPACT=HIGH)

In [3]:
m["vep_pLOF"] = m["is_pLOF_vep"] == True
m["bf4_pLOF"] = m["bf4_lof"].notna()
print(pd.crosstab(m["bf4_pLOF"], m["vep_pLOF"],
                  rownames=["BF4 LoF flag"], colnames=["VEP HIGH"]))
print(f"\nVEP pLOF total: {int(m['vep_pLOF'].sum())} | BF4 LoF total: {int(m['bf4_pLOF'].sum())}")

# os 6 BF4-LoF: o VEP concorda (HIGH)?
b_lof = m[m["bf4_pLOF"]]
print(f"\ndos {len(b_lof)} BF4-LoF, VEP=HIGH em: {int(b_lof['vep_pLOF'].sum())}")
# os pLOF extras do VEP: qual status no BF4?
extra = m[(m["vep_pLOF"]) & (~m["bf4_pLOF"])]
print(f"\npLOF que SÓ o VEP achou: {len(extra)}")
print("   status BF4 desses:", dict(extra["bf4_status"].value_counts(dropna=False)))
print("   (ultra-raras MAF<1e-4:", int((extra['maf_max']<1e-4).sum()), ")")

VEP HIGH      False  True 
BF4 LoF flag              
False           361     19
True              1      5

VEP pLOF total: 24 | BF4 LoF total: 6

dos 6 BF4-LoF, VEP=HIGH em: 5

pLOF que SÓ o VEP achou: 19
   status BF4 desses: {'not_found': np.int64(19)}
   (ultra-raras MAF<1e-4: 19 )


## 3. Concordância de consequência (nas que ambos anotaram)
Normalizamos os termos dos dois para categorias comparáveis.

In [4]:
import re
def categ(s):
    s = str(s).lower()
    if re.search(r"frameshift|stop_gained|splice_donor|splice_acceptor|stop_lost|start_lost", s): return "pLOF"
    if "missense" in s: return "missense"
    if "synonymous" in s: return "synonymous"
    if "utr" in s: return "UTR"
    if "intron" in s: return "intron"
    if s in ("nan","none",""): return None
    return "other"

both = m[(m["bf4_found"]) & (m["vep_annot"])].copy()
both["bf4_cat"] = both["bf4_conseq"].map(categ)
both["vep_cat"] = both["vep_conseq"].map(categ)
ct = pd.crosstab(both["bf4_cat"], both["vep_cat"])
print(f"variantes anotadas por AMBOS: {len(both)}")
print(ct)
agree = (both["bf4_cat"] == both["vep_cat"]).mean()
print(f"\nconcordância de categoria: {agree:.1%}")
print("(divergências comuns vêm de VEP --pick escolher transcrito diferente do dbNSFP)")

variantes anotadas por AMBOS: 97
vep_cat     UTR  intron  missense  pLOF  synonymous
bf4_cat                                            
UTR           5       0         0     0           0
intron        0      18         0     0           0
missense      0       1        45     0           0
other         0       0         0     0           1
pLOF          0       1         0     5           0
synonymous    0       3         0     0          18

concordância de categoria: 93.8%
(divergências comuns vêm de VEP --pick escolher transcrito diferente do dbNSFP)


## 4. AlphaMissense — BF4 vs VEP/dbNSFP

In [5]:
print(f"AlphaMissense scores -> BF4: {int(m['bf4_am'].notna().sum())} | VEP/dbNSFP: {int(m['vep_am'].notna().sum())}")
print("\nAlphaMissense_pred (VEP/dbNSFP):")
print(m["vep_am_pred"].value_counts(dropna=False).to_string())
print("\n(B=likely benign, A=ambiguous, P/D=likely pathogenic — depende da versão dbNSFP)")

AlphaMissense scores -> BF4: 0 | VEP/dbNSFP: 177

AlphaMissense_pred (VEP/dbNSFP):
vep_am_pred
NaN    209
B      123
P       31
A       23

(B=likely benign, A=ambiguous, P/D=likely pathogenic — depende da versão dbNSFP)


## 5. REVEL — cross-check dos scores pré-computados
Ambos derivam de dbNSFP; nas variantes que os dois têm, devem bater.

In [6]:
# REVEL do dbNSFP (ANNOVAR multianno) para o cross-check
ann = pd.read_csv(R04 / "znf175_annovar.hg38_multianno.txt", sep="\t", dtype=str, na_values=".")
snv = ann[(ann["Ref"].str.len()==1) & (ann["Alt"].str.len()==1)].copy()
snv["key"] = snv["Start"].astype(str)+":"+snv["Ref"]+":"+snv["Alt"]
snv["dbnsfp_revel"] = pd.to_numeric(snv["REVEL_score"], errors="coerce")
m2 = m.merge(snv[["key","dbnsfp_revel"]], on="key", how="left")

both_revel = m2.dropna(subset=["bf4_revel","dbnsfp_revel"])
print(f"variantes com REVEL nos dois: {len(both_revel)}")
if len(both_revel) > 1:
    print(f"correlação BF4 revel_max x dbNSFP REVEL: {both_revel['bf4_revel'].corr(both_revel['dbnsfp_revel']):.3f}")
print(f"REVEL disponível -> BF4: {int(m2['bf4_revel'].notna().sum())} | dbNSFP: {int(m2['dbnsfp_revel'].notna().sum())}")

variantes com REVEL nos dois: 45
correlação BF4 revel_max x dbNSFP REVEL: 1.000
REVEL disponível -> BF4: 45 | dbNSFP: 176


In [7]:
# --- Reconciliation table + summary ---
out_cols = ["key","CHROM","POS","REF","ALT","group","maf_max",
            "bf4_status","bf4_conseq","bf4_lof","bf4_am","bf4_revel",
            "vep_conseq","vep_impact","vep_pLOF","vep_am","vep_am_pred"]
m[out_cols].sort_values("POS").to_csv(R05 / "vep_vs_bf4_reconciliation.csv", index=False)

n=len(m); vep_cov=int(m["vep_annot"].sum()); bf4_cov=int(m["bf4_found"].sum())
nvplof=int(m["vep_pLOF"].sum()); nbplof=int(m["bf4_pLOF"].sum())
nshare=int(m[m["bf4_pLOF"]]["vep_pLOF"].sum()); nextra=int(((m["vep_pLOF"])&(~m["bf4_pLOF"])).sum())
summary = f'''# NB 05 — VEP/dbNSFP vs Biofilter 4 reconciliation

## Coverage
- VEP annotated: {vep_cov}/{n} | BF4 found: {bf4_cov}/{n}
- VEP-only (BF4 missed): {int(((~m["bf4_found"])&(m["vep_annot"])).sum())} variants
  -> of which pLOF: {int(m[(~m["bf4_found"])&(m["vep_annot"])]["is_pLOF_vep"].sum())},
     with AlphaMissense: {int(m[(~m["bf4_found"])&(m["vep_annot"])]["vep_am"].notna().sum())}

## pLOF
- VEP (IMPACT=HIGH): {nvplof} | BF4 (LOFTEE): {nbplof}
- {nshare} of {nbplof} BF4-LoF are also VEP-HIGH. The 1 difference (51573407:CAG:C) is a TRANSCRIPT-CHOICE
  artifact: frameshift on one transcript (BF4/LOFTEE) vs splice_region/intron on VEP --pick's transcript.
- extra pLOF only VEP found: {nextra} (mostly BF4 not_found, ultra-rare private burden candidates)
- NOTE: VEP --pick can UNDER-call pLOF (one transcript only); --most_severe would catch 51573407.
  For a pLOF-focused pass, re-run VEP with --most_severe.

## AlphaMissense
- BF4: {int(m["bf4_am"].notna().sum())} | VEP/dbNSFP: {int(m["vep_am"].notna().sum())}

## Verdict
VEP/dbNSFP is strictly superior for THIS task: same calls where both annotate, but ~100% coverage
(by coordinate) vs BF4's ~25%, 4x more pLOF (the private burden candidates), and AlphaMissense where
BF4 had none. IMPORTANT: BF4's low coverage here is NOT inherent — the loaded BF4 DB was filtered to
gnomAD AC >= 5, so AC < 5 variants were excluded (they exist in gnomAD; BF4-found min ac = 5). Reloading
BF4 without that cutoff would recover most not_found. BF4's standing edge = convenience + extra
precomputed scores (SpliceAI/Pangolin/CADD) for variants in its DB.

## Caveats
- VEP --pick chooses ONE transcript -> consequence disagreements with dbNSFP (e.g. intron on the picked
  transcript but missense -> AlphaMissense on another). Also under-calls pLOF (see 51573407). --most_severe fixes this.
- LOFTEE HC/LC not run (human_ancestor/gerp data absent) -> VEP pLOF here = IMPACT=HIGH, not LOFTEE-filtered.

## Output
- vep_vs_bf4_reconciliation.csv
'''
(R05 / "nb05_reconciliation_summary.md").write_text(summary)
print(summary)

# NB 05 — VEP/dbNSFP vs Biofilter 4 reconciliation

## Coverage
- VEP annotated: 384/386 | BF4 found: 97/386
- VEP-only (BF4 missed): 287 variants
  -> of which pLOF: 19,
     with AlphaMissense: 131

## pLOF
- VEP (IMPACT=HIGH): 24 | BF4 (LOFTEE): 6
- 5 of 6 BF4-LoF are also VEP-HIGH. The 1 difference (51573407:CAG:C) is a TRANSCRIPT-CHOICE
  artifact: frameshift on one transcript (BF4/LOFTEE) vs splice_region/intron on VEP --pick's transcript.
- extra pLOF only VEP found: 19 (mostly BF4 not_found, ultra-rare private burden candidates)
- NOTE: VEP --pick can UNDER-call pLOF (one transcript only); --most_severe would catch 51573407.
  For a pLOF-focused pass, re-run VEP with --most_severe.

## AlphaMissense
- BF4: 0 | VEP/dbNSFP: 177

## Verdict
VEP/dbNSFP is strictly superior for THIS task: same calls where both annotate, but ~100% coverage
(by coordinate) vs BF4's ~25%, 4x more pLOF (the private burden candidates), and AlphaMissense where
BF4 had none. IMPORTANT: BF4's low coverage h